# White light CCF of HD 189733 - two nights

In this notebook, we present a simple example of the application of the Doppler Shadow technique to two nights of ESPRESSO observations of HD 189733, a K1V star, extracting the white light CCFs provided by the DRS. The nights of observations were 11-08-2021 and 31-08-2021.

We run two different instances of HECATE and then use the multi_night_analysis class to combine the data.

In [ ]:
import numpy as np

from HECATE.HECATE import HECATE   # main class
from HECATE.get_data import *
from HECATE.multi_night_analysis import multi_night_analysis # to perform the multi-night analysis

In [ ]:
stellar_params = {
                "Teff":4969, "Teff_err":43,   #effective temperature [K]
                "logg":4.60, "logg_err":0.01, #superficial gravity [dex]
                "FeH":-0.07, "FeH_err":0.02,  #metallicity [dex]
                "P_rot":2.21857312,           #rotation period [d]
                "R_star":0.766,               #radius [solar radii]
                "inc_star":71.87              #stellar inclination [º]
                }

planet_params = {
                "P_orb":2.21857312,           #orbital period [d]
                "a_R":8.76863,                #system scale [stellar radii]
                "Rp_Rs":0.1602,               #planet-to-star radius ratio 
                "t0":53988.30339,             #mid-transit time [d]
                "e":0,                        #orbital eccentricity
                "w":90,                       #argument of periastron [º]
                "inc_planet":85.465,          #planet inclination [º]
                "lbda":-1.00,                 #spin-orbit angle [º]
                "dfp": -0.002424              #mid-transit phase shift, night of 2021-09-11
                }

Now we quickly run HECATE for the first night of observations, saving the local CCF parameters.

In [ ]:
CCFs, time, airmass, berv, bervmax, snr, list_ccfs = get_CCFs(planet_params, stellar_params,
                                                              day='2021-08-11', 
                                                              directory_path="HD189733_ESPRESSO_white_light_ccfs",
                                                              plot=False)

In [ ]:
hecate11 = HECATE(planet_params, stellar_params, time, CCFs, spectra=None, plot_soap=False)

In [ ]:
plot = {"fits_initial_CCF":False,         # plot the initial fits to the CCFs
        "sys_vel_ccf":False,              # plot the systemic velocity of the star in the CCFs
        "master_out_of_transit":False,    # plot the master out-of-transit CCF
        "local_CCFs":True,                # plot the local CCFs
        "photometrical_rescale":False     # if the local CCFs are rescaled by the photometric transit model
        }

ccf_type = "white light"
model_fit = "modified Gaussian"

In [ ]:
local_CCFs, CCFs_flux_corr, CCFs_sub_all, avg_out_of_transit_CCF = hecate11.extract_local_CCF(model_fit, plot, save=None)

In [ ]:
master_results11 = hecate11.get_profile_parameters(profiles=avg_out_of_transit_CCF, 
                                               data_type="CCF", 
                                               observation_type="master", 
                                               model="modified Gaussian", 
                                               print_output=False, 
                                               plot_fit=False)

In [ ]:
local_results11 = hecate11.get_profile_parameters(profiles=local_CCFs, 
                                              data_type="CCF", 
                                              observation_type="local", 
                                              model="modified Gaussian", 
                                              print_output=False, 
                                              plot_fit=False)

In [ ]:
indices_final11 = np.array(list(set(np.where(local_results11['R2'] >= 0.95)[0]).intersection(set(np.where(hecate11.mu_in >= 0.3)[0]))))

local_params11 = [local_results11['central_rv'], local_results11['width'], local_results11['intensity']]
master_params11 = [master_results11['central_rv'], master_results11['width'], master_results11['intensity']]

Now we do the same for the second night.

In [ ]:
planet_params["dfp"] = -0.002300    # mid-transit phase shift, night of 2021-08-31

In [ ]:
CCFs, time, airmass, berv, bervmax, snr, list_ccfs = get_CCFs(planet_params, stellar_params,
                                                              day='2021-08-31', 
                                                              directory_path="HD189733_ESPRESSO_white_light_ccfs",
                                                              plot=False)

In [ ]:
hecate31 = HECATE(planet_params, stellar_params, time, CCFs, None)

plot = {"fits_initial_CCF":False, 
        "sys_vel_ccf":False, 
        "avg_out_of_transit_CCF":False, 
        "local_CCFs":True, 
        "photometrical_rescale":False}

ccf_type = "white light"
model_fit = "modified Gaussian"

local_CCFs, CCFs_flux_corr, CCFs_sub_all, avg_out_of_transit_CCF = hecate31.extract_local_CCF(model_fit, plot, save=None)

In [ ]:
master_results31 = hecate31.get_profile_parameters(profiles=avg_out_of_transit_CCF, data_type="CCF", 
                                                    observation_type="master", model="modified Gaussian", 
                                                    print_output=False, plot_fit=False)

local_results31 = hecate31.get_profile_parameters(profiles=local_CCFs, data_type="CCF", 
                                                    observation_type="local", model="modified Gaussian", 
                                                    print_output=False, plot_fit=False)

In [ ]:
indices_final31 = np.array(list(set(np.where(local_results31['R2'] >= 0.9)[0]).intersection(set(np.where(hecate31.mu_in >= 0.3)[0]))))

local_params31 = [local_results31['central_rv'], local_results31['width'], local_results31['intensity']]
master_params31 = [master_results31['central_rv'], master_results31['width'], master_results31['intensity']]

Having the results for each night, we build the following dictionaries:

In [ ]:
night_data_11 = {
    'hecate': hecate11,                             # HECATE instance
    'indices': indices_final11,                     # indices of points to retain
    'local_params': np.array(local_params11),       # local CCF parameters 
    'master_params': np.array(master_params11),     # master CCF parameters
    'color': "blue",                                # color to use in the plots for this night
    'label': "2021-08-11"                           # label to use in the plots for this night
        }

night_data_31 = {
    'hecate': hecate31,
    'indices': indices_final31,
    'local_params': np.array(local_params31),
    'master_params': np.array(master_params31),
    'color': "red",
    'label': "2021-08-31"
        }

In [ ]:
night_data = {"2021-08-11": night_data_11,
              "2021-08-31": night_data_31}

In [ ]:
mult_night = multi_night_analysis(night_data, data_type='CCF')

We can just plot the two nights data in one single plot for each position parameter ($\phi$ or $\mu$):

In [ ]:
mult_night.plot_parameters(param_type='phases', 
                            fit_each_night=False, 
                            fit_combined=False, 
                            combined_night_names=None, 
                            plot_nested=False, 
                            suptitle="Local white light CCF parameters")


mult_night.plot_parameters(param_type='mu', 
                            fit_each_night=False, 
                            fit_combined=False, 
                            combined_night_names=None, 
                            plot_nested=False, 
                            suptitle="Local white light CCF parameters")

Or fit a linear model to each night:

In [ ]:
fit_results = mult_night.plot_parameters(param_type='phases', 
                                      fit_each_night=True, 
                                      fit_combined=False, 
                                      fit_param_indices = np.array([0]),
                                      suptitle="Local white light CCF parameters")

In [ ]:
fit_results = mult_night.plot_parameters(param_type='mu', 
                                      fit_each_night=True, 
                                      fit_combined=False, 
                                      fit_param_indices = np.array([1, 2]),
                                      suptitle="Local white light CCF parameters")